In [4]:
import os
import re
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown
from supabase import create_client, Client
from sentence_transformers import SentenceTransformer

# Load environment variables
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
LLM_URL = "http://localhost:11434/v1/chat/completions"

supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Loading BAAI/bge-small-en-v1.5 embedding model...")
model_bge = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("Setup complete.")

def call_local_llm(prompt, temperature=0.2):
    payload = {
        "model": "mistral",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature
    }
    try:
        resp = requests.post(LLM_URL, json=payload, timeout=60)
        return resp.json()['choices'][0]['message']['content']
    except Exception as e:
        print(f"Local LLM Error: {e}")
        return ""


Loading BAAI/bge-small-en-v1.5 embedding model...
Setup complete.


In [5]:
def trace_hybrid_inference_pipeline(query, v_rpc, g_table):
    print("================================================================================")
    print("MEMULAI ALUR INFERENCE")
    print(f"QUERY PENGGUNA : \"{query}\"")
    print("================================================================================")

    # --------------------------------------------------------------------------
    # TAHAP 1: MULTI-QUERY EXPANSION (MQE)
    # --------------------------------------------------------------------------
    print("\n[TAHAP 1: MULTI-QUERY EXPANSION (MQE)]")
    mq_prompt = (
        f"Analyze the user query: '{query}'. "
        "Generate 2 alternative search queries in English that use clinical synonyms, "
        "alternative medical terminology, or related clinical conditions for the core concepts "
        "Return ONLY a JSON array of strings."
    )
    
    mq_resp = call_local_llm(mq_prompt, temperature=0.3)
    queries = [query]
    try:
        expanded = json.loads(re.sub(r"```json\n?|\n?```", "", mq_resp))
        queries.extend(expanded)
    except:
        pass

    print("Hasil Ekspansi Query:")
    for idx, q in enumerate(queries[:4], 1):
        status = "(Query Asli)" if idx == 1 else f"(Ekspansi #{idx-1})"
        print(f"   {idx}. {status} : \"{q}\"")

    # --------------------------------------------------------------------------
    # TAHAP 2: JALUR A - VECTOR RETRIEVAL & GLOBAL COSINE SORTING
    # --------------------------------------------------------------------------
    print("\n[TAHAP 2: JALUR A - VECTOR RETRIEVAL & GLOBAL SORTING]")
    unique_chunks_map = {}
    mqe_results_trace = []

    for q_idx, q in enumerate(queries[:3], 1):
        try:
            q_vec = model_bge.encode([q], normalize_embeddings=True)[0].tolist()
            v_res = supabase_client.rpc(v_rpc, {
                "query_embedding": q_vec,
                "match_threshold": 0.05,
                "match_count": 5
            }).execute()

            if v_res.data:
                for d in v_res.data:
                    c_id = d.get('id', 'N/A')
                    doc_text = d['text_content'].strip()
                    sim_score = float(d.get('similarity', 0.0))
                    
                    mqe_results_trace.append({
                        "Query Origin": f"Q{q_idx}",
                        "Chunk ID": str(c_id)[:8] + "...",
                        "Cosine Similarity": f"{sim_score:.4f}",
                        "Text Snippet": doc_text[:60] + "..."
                    })
                    
                    if doc_text in unique_chunks_map:
                        if sim_score > unique_chunks_map[doc_text]['score']:
                            unique_chunks_map[doc_text] = {'id': c_id, 'score': sim_score}
                    else:
                        unique_chunks_map[doc_text] = {'id': c_id, 'score': sim_score}
        except Exception as e:
            print(f"Error Vector Retrieval: {e}")

    if mqe_results_trace:
        print("\nTabel Hasil Pencarian Vektor (Raw):")
        display(pd.DataFrame(mqe_results_trace))
    else:
        print("\nTidak ada data dokumen yang ditemukan.")

    sorted_chunks = sorted(unique_chunks_map.items(), key=lambda item: item[1]['score'], reverse=True)
    top_5_chunks = sorted_chunks[:5]
    
    top_5_trace = []
    v_docs = []
    for r_idx, (c_text, c_data) in enumerate(top_5_chunks, 1):
        top_5_trace.append({
            "Rank": f"Top-{r_idx}",
            "Chunk ID": str(c_data['id'])[:8] + "...",
            "Cosine Similarity": f"{c_data['score']:.4f}",
            "Text Snippet": c_text[:100] + "..."
        })
        v_docs.append(c_text)

    if top_5_trace:
        print("\nTop 5 Chunk Hasil Global Sorting:")
        display(pd.DataFrame(top_5_trace))

    # --------------------------------------------------------------------------
    # TAHAP 3: JALUR B - GRAPH RETRIEVAL
    # --------------------------------------------------------------------------
    print("\n[TAHAP 3: JALUR B - GRAPH RETRIEVAL]")
    all_g_rels = []
    for q_idx, q in enumerate(queries[:3], 1):
        ent_prompt = (
            f"Extract the specific medical entities (ONLY diseases, symptoms, or plant names) from the following text: '{q}'. "
            "Translate them to English if they are in another language. "
            "EXCLUDE generic terms like 'herbal', 'medicine', 'remedy', 'obat', 'treatment', 'cure', 'natural'. "
            "Do NOT guess plant names if they are not explicitly in the text. "
            "Return ONLY a JSON array of strings (e.g., ['common cold', 'indigestion'])."
        )
        ent_resp = call_local_llm(ent_prompt, temperature=0.1)
        try:
            entities = json.loads(re.sub(r"```json\n?|\n?```", "", ent_resp))
        except:
            entities = []

        print(f"Entitas Terdeteksi dari Query #{q_idx}: {entities}")

        for ent in entities:
            safe_ent = re.sub(r"[^a-zA-Z0-9\s]", "", str(ent).lower().strip())
            if len(safe_ent) < 3: continue
            
            # Additional check to prevent overly generic or short matches
            if safe_ent in ["herb", "herbal", "medicine", "remedy", "cure", "drug", "obat"]: continue
            
            try:
                res_g = supabase_client.table(g_table).select('*')\
                    .or_(f"entity_1.ilike.%{safe_ent}%,entity_2.ilike.%{safe_ent}%")\
                    .execute()
                for row in res_g.data:
                    rel_str = f"({row['entity_1']} -> [{str(row['relation']).replace('_',' ')}] -> {row['entity_2']})"
                    all_g_rels.append({'id': row.get('id', 'N/A'), 'triplet': rel_str})
            except Exception as e:
                print(f"Error Graph Retrieval pada entitas '{safe_ent}': {e}")
                
    unique_rels_dict = {}
    for r in all_g_rels:
        if r['triplet'] not in unique_rels_dict:
            unique_rels_dict[r['triplet']] = r['id']
            
    final_g_rels = [{'Graph ID': r_id, 'Triplet': r_trip} for r_trip, r_id in unique_rels_dict.items()]
    top_10_graphs = final_g_rels[:10]
    final_g_rels_text = [r['Triplet'] for r in top_10_graphs]

    print(f"\nTotal Graph Triplets ditemukan (Setelah Deduplikasi): {len(final_g_rels_text)} (Top 10 diambil)")
    for g_idx, rel in enumerate(final_g_rels_text, 1):
        print(f"   {g_idx}. {rel}")

    # --------------------------------------------------------------------------
    # TAHAP 4: CONTEXT FUSION (PROMPT ASSEMBLY)
    # --------------------------------------------------------------------------
    print("\n[TAHAP 4: CONTEXT FUSION (PROMPT ASSEMBLY)]")
    context_vector_str = "\n".join([f"[Doc {i+1}]: {doc}" for i, doc in enumerate(v_docs)])
    context_graph_str = "\n".join([f"[Graph Fact {i+1}]: {rel}" for i, rel in enumerate(final_g_rels_text)])

    final_prompt = f"""
    Context 1 (Scientific Relations):
    {context_graph_str if context_graph_str else "No graph relations available."}

    Context 2 (Research Abstracts):
    {context_vector_str if context_vector_str else "No research abstracts available."}

    Question: {query}

    Instructions:
    1. Identify the specific herb mentioned in the context.
    2. Describe its biological properties (e.g., anti-inflammatory, antioxidant) ONLY if stated.
    3. Write a cohesive 4-6 sentence paragraph.
    4. If a detail is not in the contexts, do not include it.
    5. Ensure your answer is as detailed as a formal medical abstract.

    Answer (English):"""
    
    print("Mempersiapkan Prompt Akhir yang akan dikirim ke LLM:\n")
    print("-" * 80)
    print(final_prompt)
    print("-" * 80)

    # --------------------------------------------------------------------------
    # TAHAP 5: LLM GENERATION (JAWABAN AKHIR)
    # --------------------------------------------------------------------------
    print("\n[TAHAP 5: FINAL AI GENERATION]")
    ans = call_local_llm(final_prompt, temperature=0.2)
    
    print("\n[JAWABAN AKHIR SISTEM HYBRID GRAPHRAG]:")
    display(Markdown(f"### Q: {query}"))
    display(Markdown("**Answer:**"))
    display(Markdown(ans))
    print("\n================================================================================")


In [6]:
print("\n" + "="*80)
print("SISTEM INFERENCE MANUAL HYBRID GRAPHRAG")
print("="*80)
print("Ketik 'exit' atau 'keluar' untuk menghentikan program.")

while True:
    print("\n" + "-"*80)
    query_input = input("Masukkan pertanyaan/query Anda: ")
    
    if query_input.lower() in ['exit', 'keluar', 'quit']:
        print("Keluar dari program.")
        break
        
    if not query_input.strip():
        print("Query tidak boleh kosong.")
        continue
        
    try:
        trace_hybrid_inference_pipeline(
            query=query_input,
            v_rpc="match_chunk_sentence",
            g_table="graph_sentence"
        )
    except Exception as e:
        print(f"Terjadi kesalahan: {e}")


SISTEM INFERENCE MANUAL HYBRID GRAPHRAG
Ketik 'exit' atau 'keluar' untuk menghentikan program.

--------------------------------------------------------------------------------
MEMULAI ALUR INFERENCE
QUERY PENGGUNA : "herb for headache"

[TAHAP 1: MULTI-QUERY EXPANSION (MQE)]
Hasil Ekspansi Query:
   1. (Query Asli) : "herb for headache"
   2. (Ekspansi #1) : "natural remedy for cephalalgia"
   3. (Ekspansi #2) : "botanical treatment for migraine"

[TAHAP 2: JALUR A - VECTOR RETRIEVAL & GLOBAL SORTING]

Tabel Hasil Pencarian Vektor (Raw):


,Query Origin,Chunk ID,Cosine Similarity,Text Snippet
0,Q1,7ee562dd...,0.7538,The cinnamon capsules contained 600 mg of drie...
1,Q1,28287c86...,0.7504,to be implicated in the pathogenic mechanisms ...
2,Q1,d43b71c6...,0.7431,Although this study was the first randomised d...
3,Q1,fb05c3ea...,0.7431,Although this study was the first randomised d...
4,Q1,bd5276d6...,0.7181,It was calcu- lated that 50 participants (i.e....
5,Q2,ff2258b4...,0.7084,"Stabilize the number of lymphocytes, NK Cells,..."
6,Q2,c8282162...,0.6907,"Therefore, additional studies with longer peri..."
7,Q2,d31aeefd...,0.6864,ABSTRACT Lakum fruit (Cayratia trifolia L. Dom...
8,Q2,7ee562dd...,0.6861,The cinnamon capsules contained 600 mg of drie...
9,Q2,15abef63...,0.6848,ABSTRACT Ginger and celery extracts are recogn...



Top 5 Chunk Hasil Global Sorting:


,Rank,Chunk ID,Cosine Similarity,Text Snippet
0,Top-1,d43b71c6...,0.7717,Although this study was the first randomised d...
1,Top-2,28287c86...,0.7690,to be implicated in the pathogenic mechanisms ...
2,Top-3,068ed740...,0.7683,While the use of traditional medicine has been...
3,Top-4,5590af40...,0.7614,ANALYTICAL DESCRIPTION OF THE MEDICINAL PLANTS...
4,Top-5,7ee562dd...,0.7538,The cinnamon capsules contained 600 mg of drie...



[TAHAP 3: JALUR B - GRAPH RETRIEVAL]
Entitas Terdeteksi dari Query #1: ['headache']
Entitas Terdeteksi dari Query #2: ['cephalalgia']
Entitas Terdeteksi dari Query #3: ['migraine']

Total Graph Triplets ditemukan (Setelah Deduplikasi): 10 (Top 10 diambil)
   1. (Calophyllum inophyllum -> [cures] -> Headaches)
   2. (Vacha -> [cures] -> Headaches)
   3. (Angelica -> [cures] -> headaches)
   4. (Ziziphus jujuba var. spinosae -> [cures] -> headaches)
   5. (Calophyllum inophyllum -> [cures] -> tension-type headache)
   6. (Acorus calamus -> [cures] -> tension-type headache)
   7. (chronic headache -> [causes] -> frequency)
   8. (chronic headache -> [causes] -> severity)
   9. (Cinnamomum zeylanicum -> [cures] -> headache)
   10. (cinnamon -> [cures] -> migraine)

[TAHAP 4: CONTEXT FUSION (PROMPT ASSEMBLY)]
Mempersiapkan Prompt Akhir yang akan dikirim ke LLM:

--------------------------------------------------------------------------------

    Context 1 (Scientific Relations):
    [Grap

### Q: herb for headache

**Answer:**

**Herb for Headache: Cinnamon (*Cinnamomum zeylanicum*)**

Cinnamon (*Cinnamomum zeylanicum*), a widely used spice, has been investigated for its therapeutic potential in alleviating headache disorders, particularly migraine. Clinical evidence from a randomized double-blind trial demonstrated that cinnamon consumption significantly reduced the frequency, severity, and duration of migraine attacks, alongside lowering serum levels of inflammatory markers such as IL-6 and nitric oxide (NO) (Doc 1). Preclinical studies further support its neuroprotective and anti-neuroinflammatory properties, attributed to bioactive compounds like cinnamaldehyde, which may mitigate vasospasm and oxidative stress implicated in migraine pathogenesis (Doc 2). The herb’s safety profile and accessibility have positioned it as a complementary therapy, though limitations such as small sample sizes and short intervention durations warrant further validation in larger, long-term trials (Doc 1). Additionally, cinnamon’s role in modulating inflammatory pathways underscores its potential as an adjunctive treatment for headache disorders (Doc 2). While traditional use of cinnamon for headaches is documented, rigorous clinical trials remain essential to confirm its efficacy and optimize dosing strategies (Doc 2).



--------------------------------------------------------------------------------
Keluar dari program.
